##### Imports and Install


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import numpy as np
import os

In [9]:
#pip install torch torchvision

### Recreate Architecture

##### MobileNetv2

In [ ]:
NUM_CLASSES = 4

mobilenetv2 = models.mobilenet_v2(weights=None)

for param in mobilenetv2.parameters():
    param.requires_grad = False

for param in mobilenetv2.features[-3:].parameters():
    param.requires_grad = True

in_features                = mobilenetv2.classifier[1].in_features
mobilenetv2.classifier[1]  = nn.Linear(in_features, NUM_CLASSES)

mobilenetv2.load_state_dict(
    torch.load('I:/Capstone/Model/mobilenetv2_trained_cpu.pth', map_location='cpu')
)

mobilenetv2.eval()
print("MobileNetV2 loaded successfully")

#### Verify Model Scores

In [ ]:
dummy_image  = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
mean         = np.array([0.485, 0.456, 0.406])
std          = np.array([0.229, 0.224, 0.225])
float_image  = dummy_image.astype(np.float32) / 255.0
normalised   = (float_image - mean) / std
chw_image    = normalised.transpose(2, 0, 1)
input_tensor = torch.from_numpy(chw_image).unsqueeze(0).float()

with torch.no_grad():
    output = mobilenetv2(input_tensor)
    print(f"Output values : {output}")
    print(f"Max value     : {output.max().item():.4f}")
    print(f"Min value     : {output.min().item():.4f}")
    print("\nIf values are between -3 and +3 proceed to Cell 4")

Output shape : torch.Size([1, 4])
Raw outputs  : tensor([[-10.5876, -16.8394,  -6.9434, -36.5163]])
Predicted class index : 2
Model verified successfully


#### Trace + Save as PTL

In [ ]:
traced_model = torch.jit.trace(mobilenetv2, input_tensor)
traced_model.eval()

# MobileNetV2 can safely use optimize_for_mobile unlike ResNet50
from torch.utils.mobile_optimizer import optimize_for_mobile
optimised_model = optimize_for_mobile(traced_model)

ptl_path = 'I:/Capstone/Model/mobilenetv2.ptl'
optimised_model._save_for_lite_interpreter(ptl_path)

size_mb = os.path.getsize(ptl_path) / (1024 * 1024)
print(f"Model saved to : {ptl_path}")
print(f"File size      : {size_mb:.2f} MB")

Output shape  : torch.Size([1, 4])
Output values : tensor([[-10.2413, -17.1350,  -7.9530, -36.9256]])
Max value     : -7.952953815460205
Min value     : -36.92555618286133


#### Verify the Saved File

In [ ]:
loaded_model = torch.jit.load(ptl_path)
loaded_model.eval()

with torch.no_grad():
    output = loaded_model(input_tensor)
    print(f"Output values : {output}")
    print(f"Max value     : {output.max().item():.4f}")
    print(f"Min value     : {output.min().item():.4f}")
    print("\nIf values match Cell 3 the PTL file is correct and ready for Android")

Model saved from CPU successfully
